# Write Rates in HDF5's Format
### Import libraries

In [1]:
import os
import sys
import pickle
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

Visualization libraries

In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
# %matplotlib qt

Global paths

In [3]:
# Current Working Directory
CWD = os.getcwd()
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))

Case-specific paths

In [4]:
case_suffix = '/O3_UMN/Inel/configs/config.06/run.01/'

In [5]:
from surqct import config as cfg

# Output directory
out_dir = cfg.HDF5Dtb + '/SurQCT/' + case_suffix
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initialize PyCOMET model
Working case

In [6]:
case_dir = WORKSPACE + '/ML/PyCOMET/run/SurQCT/' + case_suffix
sys.path.append(os.path.abspath(case_dir))

In [35]:
os.path.abspath(case_dir)

'/home/zanardi/WORKSPACE/ML/PyCOMET/run/SurQCT/O3_UMN/Inel/configs/config.06/run.01'

In [18]:
path = os.path.normpath(case_dir)
path.split(os.sep)
path

'/home/zanardi/WORKSPACE/ML/PyCOMET/run/SurQCT/O3_UMN/Inel/configs/config.06/run.01'

In [33]:
from pathlib import Path
x = np.array(Path(os.path.normpath(case_dir)).parts)
idx = np.where(x == 'SurQCT')
string = list(x[int(idx[0])+1:])
'/'.join(string)

'O3_UMN/Inel/configs/config.06/run.01'

In [7]:
os.chdir(case_dir)

Import additional libraries

In [8]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [9]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)


[PyCOMET]: Initializing 'OpNN' network ...

[PyCOMET]: Building 'OpNN' network ...
Model: "SurQCT_OpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
normalization (Normalization)   (None, 5)            11          input_1[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 5)            11          input_2[0][0]                    
____

In [10]:
os.chdir(CWD)

### Initialize SurQCT parameters
Import additional libraries

In [11]:
from surqct.src.chemical_systems import systems as syst
from surqct.src.rates_type.rates_type import get_rates_type_mod

SurQCT model inputs

In [12]:
surqct_inp = WORKSPACE + '/ML/PyCOMET/run/SurQCT/O3_UMN/Inel/configs/datagen/InputData.py'
sys.path.append(os.path.dirname(surqct_inp))

import InputData

In [13]:
TTranVec = [1500.]

Initializing chemical system and rates type module

In [14]:
System = syst.get_system(
    NameLong=InputData.SystemName,
    ProcNames=InputData.RatesType
)
# Initializing rates type module
rates_type_mod = get_rates_type_mod(System)

### Writing rates

In [16]:
rates_type_mod.save_predictiondata(
    InputData=InputData,
    System=System,
    Model=model.net,
    TTranVec=TTranVec,
    PathToHDF5Fld=out_dir,
    Columns=inputs.data.columns['x']
)

  0%|          | 0/6114 [00:00<?, ?it/s]

[SurQCT]:   Generating Inel Rate Matrix
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_rIn', 'log_rOut', 'log_rOut-rIn', 'rIn/rMin_min', 'log_rOut/rMin_min']


100%|██████████| 6114/6114 [08:20<00:00, 12.22it/s]


In [17]:
# for T in InputData.TTranVecTest:
#     rates_type_mod.write_predictiondata(
#         InputData=InputData,
#         System=System,
#         Model=model.net,
#         KineticFldr=out_dir,
#         TTran=T,
#         Columns=inputs.data.columns['x']
#     )